# 1. От баланса фотонов к измеряемому сигналу

Начинаем с одного мгновенно испущенного фотона и однородной среды.
После этой работы должны быть понятны размерности уравнения переноса,
нормировка фазовой функции и различие плотности фотонов, скалярного потока
и ожидаемого числа регистраций. Все числа ниже — учебные параметры.

In [ ]:
from pathlib import Path
import sys, time, platform
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Notebook can be started from the repo root or notebooks/course.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/lighthit').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the LightHit repository')
sys.path.insert(0, str(ROOT / 'src'))
from lighthit import Medium, SolverSettings, PointGreenSolver

# Explicit public test medium. No private provider is imported.
medium = Medium(0.04, 0.05, 0.7, 1.35, 450.0, 'course-synthetic')
np.set_printoptions(precision=7, suppress=True)
print('Python:', sys.executable)
print('Platform:', platform.platform())

## 1.1. Что находится в элементе фазового пространства

Пусть $f(\mathbf x,\mathbf s,t)\,d^3x\,d\Omega$ — ожидаемое число фотонов,
где $\mathbf s$ — единичное направление. Скорость $v=c_0/n_g$, интенсивность
$I=vf$. За $dt$ фотон смещается на $v\mathbf s\,dt$; вероятность поглощения
равна $v\mu_a\,dt$, рассеяния — $v\mu_s\,dt$ с точностью до $O(dt^2)$.

Вычитая уход и добавляя приход из всех направлений, получаем
$$
\frac1v\partial_t I+\mathbf s\cdot\nabla I+
(\mu_a+\mu_s)I-\mu_s\int p(\mathbf s\cdot\mathbf s')I(\mathbf s')d\Omega'=q.
$$
$\mu_a,\mu_s$ имеют размерность м$^{-1}$; $p$ нормирована интегралом по сфере
к единице. $q\,d^3x\,dt\,d\Omega$ — число испущенных фотонов.

Для HG-функции
$$p(x)=\frac{1-g^2}{4\pi(1+g^2-2gx)^{3/2}},\quad -1<g<1.$$
Проверим нормировку и средний косинус: $2\pi\int p(x)dx=1$,
$2\pi\int xp(x)dx=g$.

In [ ]:
from scipy.special import roots_legendre
from lighthit.single import hg_phase
x, w = roots_legendre(256)
p = hg_phase(x, medium.g)
norm = 2*np.pi*np.dot(w,p)
mean = 2*np.pi*np.dot(w,x*p)
print('normalization =', norm, 'mean cosine =', mean)
assert abs(norm-1) < 1e-11 and abs(mean-medium.g) < 1e-11
fig,ax=plt.subplots();ax.plot(x,p);ax.set(xlabel='cos(scattering angle)',ylabel='p [sr^-1]');plt.show()

## 1.2. Закон сохранения

Интеграл члена рассеяния по выходному направлению равен
$\mu_s\int I\,d\Omega$: он сокращает уход из направления.
После интегрирования также по всему пространству поток на бесконечности
исчезает. Для одиночной вспышки:
$$N(t)=\int f\,d^3x\,d\Omega=e^{-\mu_a vt}.$$
Сохранение $N$ при $\mu_a=0$ не означает, что плотность в конкретной точке
не меняется. Рассеяние переносит её в другие точки и направления.

In [ ]:
t=np.linspace(0,500,301)
N=np.exp(-medium.absorption_per_m*medium.speed_m_per_ns*t)
fig,ax=plt.subplots();ax.plot(t,N);ax.set(xlabel='t [ns]',ylabel='surviving photon number');plt.show()

## 1.3. Сигнал точечного изотропного приёмника

Определяем $K(\mathbf r,t)=\int I(\mathbf r,\mathbf s,t)d\Omega$.
Это отклик на единицу эффективной площади: м$^{-2}$ нс$^{-1}$.
Заряд $Q=\int Kdt$ имеет размерность м$^{-2}$. Для малого приёмника с
эффективной площадью $A_{\rm eff}$ математическое ожидание регистраций —
$A_{\rm eff}Q$. Никакая площадь не приравнивается неявно единице.

Для изотропной вспышки прямой свет:
$$K^{(0)}(r,t)=\frac{e^{-\mu_t r}}{4\pi r^2}\delta(t-r/v),\quad \mu_t=\mu_a+\mu_s.$$
Для направленной вспышки свет остаётся на луче. Значение в точном направлении
луча у точечного приёмника сингулярно; код требует иной геометрии.

In [ ]:
r=np.array([10.,20.,40.]);front=r/medium.speed_m_per_ns
Q0=np.exp(-medium.extinction_per_m*r)/(4*np.pi*r*r)
print('r [m], front [ns], Q0 [m^-2]')
print(np.column_stack([r,front,Q0]))
# Integration over time bins containing the delta peak.
edges=np.arange(0,251,5.)
bins=np.zeros((len(edges)-1,len(r)))
for d in range(len(r)):
    j=np.searchsorted(edges,front[d],side='right')-1
    if 0<=j<len(bins): bins[j,d]=Q0[d]
assert np.allclose(bins.sum(0),Q0)

## Задания

1. Восстановить вывод RTE из баланса в прямоугольном объёме; указать порядок
отброшенных по $dt$ величин. Проверить размерность каждого члена.
2. Получить формулу $K^{(0)}$ из $v e^{-\mu_tvt}\delta^3(\mathbf r-vt\mathbf s_0)$
усреднением по исходному направлению, не потеряв множитель $v$.
3. Объяснить, почему $\int\mathbf s I\,d\Omega$ и $\int I\,d\Omega$ не взаимозаменяемы.

Код: `medium.py`, `single.py`; текст книги: главы 1 и 2.